# Praktikum 1.2 — Mempelajari Penggunaan Driver Neo4j


## 1. Impor library dan siapkan koneksi

In [1]:
from neo4j import GraphDatabase, RoutingControl
from getpass import getpass

# URI sesuai contoh pada modul
URI = "neo4j://localhost:7687"
USERNAME = "neo4j"

# Masukkan password database Neo4j saat diminta.
# Password tidak ditulis langsung di notebook agar tidak tersimpan di file.
PASSWORD = getpass("Masukkan password Neo4j: ")

# Nama database sesuai contoh pada modul
DATABASE = "neo4j"

## 2. Membuat fungsi `add_friend`

Fungsi ini membuat node untuk dua orang (jika belum ada), lalu membuat relasi `KNOWS` dari orang pertama ke temannya. `MERGE` digunakan agar data yang sama tidak dibuat berulang kali.

In [2]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name,
        friend_name=friend_name,
        database_=DATABASE,
    )

## 3. Membuat fungsi `print_friends`

Fungsi berikut mencari semua node `Person` yang terhubung dari orang yang dipilih melalui relasi `KNOWS`, kemudian mencetak nama teman secara alfabetis.

In [3]:
def print_friends(driver, name):
    records, summary, keys = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) "
        "WHERE a.name = $name "
        "RETURN friend.name AS friend_name "
        "ORDER BY friend.name",
        name=name,
        database_=DATABASE,
        routing_=RoutingControl.READ,
    )

    if not records:
        print(f"Belum ada teman yang tercatat untuk {name}.")
        return

    for record in records:
        print(record["friend_name"])

## 4. Jalankan contoh pada modul

In [5]:
with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    # Memeriksa apakah koneksi berhasil
    driver.verify_connectivity()
    print("Koneksi ke Neo4j berhasil.")

    # Membuat relasi pertemanan sesuai contoh modul
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")

    # Menampilkan teman Arthur
    print("Teman Arthur:")
    print_friends(driver, "Arthur")

Koneksi ke Neo4j berhasil.
Teman Arthur:
Guinevere
Lancelot
Merlin


## 5. Pertanyaan Pendahuluan (Checkpoint)

Setiap jawaban di bawah dibuktikan dengan query yang dijalankan pada database yang sama.

### Checkpoint 1: Apa nama database yang digunakan pada tutorial ini?

**Jawaban:** database bernama **`neo4j`**. Nama ini terlihat pada parameter `database_="neo4j"` di `execute_query` (pada notebook ini dipegang variabel `DATABASE`). `neo4j` adalah database *default* pada instance Neo4j. Selain itu ada database `system` yang khusus untuk administrasi (membuat atau menghapus database, mengatur pengguna) dan tidak dipakai untuk menyimpan data.

Bukti lewat query:

In [6]:
with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    records, _, _ = driver.execute_query(
        "CALL db.info() YIELD name RETURN name", database_=DATABASE
    )
    print("Database yang sedang dipakai:", records[0]["name"])

    records, _, _ = driver.execute_query(
        "SHOW DATABASES YIELD name, default, currentStatus", database_="system"
    )
    print("\nDaftar database pada instance ini:")
    for r in records:
        tanda = "  <- database default" if r["default"] else ""
        print(f"  - {r['name']:10s} status = {r['currentStatus']}{tanda}")

Database yang sedang dipakai: neo4j

Daftar database pada instance ini:
  - neo4j      status = online  <- database default
  - system     status = online


### Checkpoint 2: Akses lain (sintaks DDL/DML Cypher) yang dapat mengakses database lain dalam 1 instance

**Jawaban:**

| Cara | Sintaks | Jenis |
|---|---|---|
| Klausa `USE` di dalam query | `USE namadb MATCH (n) RETURN count(n)` | DML (membaca/menulis data pada database tertentu) |
| Perintah Neo4j Browser | `:use namadb` | Perintah Browser (memilih database aktif) |
| Melihat daftar database | `SHOW DATABASES` (dijalankan pada database `system`) | DDL administrasi |
| Membuat database | `CREATE DATABASE namadb` | DDL administrasi |
| Menghapus database | `DROP DATABASE namadb` | DDL administrasi |
| Dari Python | `driver.execute_query(..., database_="namadb")` atau `driver.session(database="namadb")` | Driver |

pada Neo4j *Community Edition* hanya tersedia satu database pengguna (`neo4j`) ditambah `system`, sehingga `CREATE DATABASE` baru bisa dipakai pada edisi Enterprise atau AuraDB. Klausa `USE` tetap bisa dijalankan pada database `neo4j`, seperti contoh berikut:

In [7]:
with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    records, _, _ = driver.execute_query(
        "USE neo4j MATCH (p:Person) RETURN count(p) AS jumlah_person",
        database_=DATABASE,
    )
    print("USE neo4j MATCH (p:Person) RETURN count(p)  ->", records[0]["jumlah_person"], "node Person")

USE neo4j MATCH (p:Person) RETURN count(p)  -> 4 node Person


### Checkpoint 3: Apa jenis graf pertemanan yang telah dibuat di sini?

**Jawaban:** graf yang dibuat adalah:

- **Berarah (*directed*)**: `(Arthur)-[:KNOWS]->(teman)` berarti Arthur mengenal mereka, belum tentu sebaliknya. Tidak ada pasangan relasi timbal-balik.
- **Tidak berbobot (*unweighted*)**: relasi `KNOWS` tidak memiliki properti bobot.
- **Homogen / unipartit**: hanya satu jenis simpul (label `Person`) dan satu jenis relasi (`KNOWS`).
- **Graf sederhana**: tidak ada relasi ganda dan tidak ada *self-loop*, karena `MERGE` tidak membuat duplikat.
- **Berbentuk bintang (*star*)**: Arthur sebagai pusat dengan derajat keluar 3, sedangkan Guinevere, Lancelot, dan Merlin masing-masing berderajat masuk 1 dan derajat keluar 0.

Bukti lewat query (hanya node `Person`, sehingga data RetailRocket pada database yang sama tidak ikut terhitung):

In [8]:
with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    # Derajat keluar dan masuk tiap Person
    records, _, _ = driver.execute_query(
        "MATCH (p:Person) "
        "OPTIONAL MATCH (p)-[o:KNOWS]->() "
        "WITH p, count(o) AS derajat_keluar "
        "OPTIONAL MATCH (p)<-[i:KNOWS]-() "
        "RETURN p.name AS nama, derajat_keluar, count(i) AS derajat_masuk "
        "ORDER BY nama",
        database_=DATABASE,
    )
    print(f"{'nama':12s} {'derajat_keluar':>15s} {'derajat_masuk':>14s}")
    for r in records:
        print(f"{r['nama']:12s} {r['derajat_keluar']:15d} {r['derajat_masuk']:14d}")

    # Ada relasi timbal-balik? (A->B dan B->A)
    r1, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(b:Person)-[:KNOWS]->(a) RETURN count(*) AS timbal_balik",
        database_=DATABASE,
    )
    # Ada properti (bobot) pada relasi?
    r2, _, _ = driver.execute_query(
        "MATCH (:Person)-[r:KNOWS]->(:Person) "
        "RETURN count(r) AS total, count(CASE WHEN size(keys(r)) > 0 THEN 1 END) AS berproperti",
        database_=DATABASE,
    )
    # Jenis relasi antar Person
    r3, _, _ = driver.execute_query(
        "MATCH (:Person)-[r]->(:Person) RETURN type(r) AS jenis, count(*) AS jumlah",
        database_=DATABASE,
    )
    print("\nRelasi timbal-balik  :", r1[0]["timbal_balik"], "(0 = berarah murni)")
    print("Relasi berproperti   :", r2[0]["berproperti"], "dari", r2[0]["total"], "(0 = tidak berbobot)")
    print("Jenis relasi         :", {r["jenis"]: r["jumlah"] for r in r3})

nama          derajat_keluar  derajat_masuk
Arthur                     3              0
Guinevere                  0              1
Lancelot                   0              1
Merlin                     0              1

Relasi timbal-balik  : 0 (0 = berarah murni)
Relasi berproperti   : 0 dari 3 (0 = tidak berbobot)
Jenis relasi         : {'KNOWS': 3}
